In [ ]:
# ── MLOps bootstrap (auto-injected by inject_mlops_cell.py) ──────────────────
import os, warnings, mlflow
warnings.filterwarnings("ignore")

SEED = 42
import random, numpy as np
random.seed(SEED)
np.random.seed(SEED)
try:
    import torch; torch.manual_seed(SEED)
except ImportError:
    pass
try:
    import tensorflow as tf; tf.random.set_seed(SEED)
except ImportError:
    pass

_nb_name = os.path.basename(os.path.abspath("__file__") if "__file__" in dir() else ".").replace(".ipynb","")
mlflow.set_tracking_uri("sqlite:///" + str(Path(__file__).parent.parent.parent / "mlflow.db")
                        if "__file__" in dir() else "sqlite:///mlflow.db")
_exp = mlflow.set_experiment(_nb_name or "unnamed_notebook")
print(f"MLflow experiment: {_exp.name}")


# PDF Q&A Assistant — Retrieval-Augmented Generation (RAG)

## 1. Project Overview

**Task:** Build a question-answering system that reads PDF documents and answers user questions with **cited context** from the source material.

**Approach:** Retrieval-Augmented Generation (RAG) — instead of relying solely on the LLM's training data, we:
1. Load PDFs and split them into chunks
2. Embed each chunk into a vector store
3. At query time, retrieve the most relevant chunks
4. Feed those chunks + the question to the LLM for a grounded answer

**Stack:**
- `LangChain` — orchestration (document loading, splitting, retrieval chain)
- `ChromaDB` — vector store for embedding storage and retrieval
- `HuggingFace Embeddings` — local embedding model (no API key needed)
- `Ollama` + `qwen3.5:9b` — local LLM for answer generation

> **No cloud APIs required.** Everything runs locally on your machine.

## 2. Learning Goals

| # | Skill |
|---|-------|
| 1 | Understand **why RAG exists** — what problem it solves that plain LLMs cannot |
| 2 | Load and parse PDF documents using LangChain's `PyPDFLoader` |
| 3 | Split documents into chunks with `RecursiveCharacterTextSplitter` |
| 4 | Create vector embeddings and store them in ChromaDB |
| 5 | Build a retrieval chain that fetches relevant context |
| 6 | Compare **naive prompting** (no context) vs **RAG** (with retrieved context) |
| 7 | Add source citations to generated answers |
| 8 | Experiment with chunk size, overlap, and embedding model choices |

## 3. Problem Statement

Imagine you have a collection of technical documents (research papers, company policies, manuals). You want to ask questions and get **accurate, grounded answers** — not hallucinations.

A plain LLM will:
- Hallucinate facts that sound plausible but aren't in your documents
- Have no knowledge of your private/proprietary content
- Lack the ability to cite specific sources

**RAG solves this** by anchoring the LLM's responses to your actual documents.

## 4. Why This Project Matters

- **RAG is the #1 pattern in production AI apps** — used by ChatGPT (with browsing), Copilot, Perplexity, and enterprise tools
- **No fine-tuning needed** — works with any LLM out of the box
- **Updatable knowledge** — add or remove documents without retraining
- **Auditable** — every answer can cite its source chunks
- **Cost-effective** — retrieval is cheap; only relevant context is sent to the LLM

## 5. How RAG Works — Conceptual Overview

RAG has two phases: **indexing** (offline, one-time) and **querying** (online, per question).

### Indexing Phase (build the knowledge base)
```
PDF files
   │
   ▼
[Load] ─── PyPDFLoader reads pages as text
   │
   ▼
[Split] ── RecursiveCharacterTextSplitter → overlapping chunks
   │
   ▼
[Embed] ── HuggingFace model → dense vectors (768-dim)
   │
   ▼
[Store] ── ChromaDB stores vectors + original text
```

### Query Phase (answer a question)
```
User Question
   │
   ├──▶ [Embed question] ── same embedding model
   │         │
   │         ▼
   │    [Retrieve top-k chunks] ── cosine similarity search in ChromaDB
   │         │
   │         ▼
   └──▶ [LLM Prompt] ── "Given this context: {chunks}, answer: {question}"
              │
              ▼
        Grounded Answer + Source Citations
```

### Key Insight

The embedding model converts both documents and questions into the **same vector space**. Similar meanings end up as nearby vectors, so retrieval finds semantically relevant chunks — even when the exact words don't match.

## 6. Document Overview (Sample PDFs)

For this notebook, we create **sample PDF documents** about machine learning topics. This makes the notebook fully self-contained — no external downloads needed.

| PDF | Topic | Pages |
|-----|-------|-------|
| `ml_fundamentals.pdf` | Supervised/unsupervised learning, bias-variance tradeoff, cross-validation | 3 |
| `neural_networks.pdf` | Perceptrons, backpropagation, activation functions, CNNs, RNNs, transformers | 3 |
| `deployment_guide.pdf` | Model serving, monitoring, A/B testing, MLOps best practices | 2 |

In a real project, you would replace these with your own PDFs (research papers, legal docs, manuals, etc.).

## 7. Environment Setup

In [ ]:
# Uncomment if any package is missing
# !pip install -q langchain langchain-ollama langchain-community langchain-huggingface
# !pip install -q langchain-text-splitters chromadb pypdf fpdf2 sentence-transformers

print("Dependencies: langchain, langchain-ollama, chromadb, pypdf, fpdf2, sentence-transformers")
print("LLM: Ollama with qwen3.5:9b (must be running locally)")

## 8. Imports

In [ ]:
import os
import warnings
import textwrap
import time
from pathlib import Path

os.environ["USE_TF"] = "0"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
warnings.filterwarnings("ignore")

from langchain_ollama import ChatOllama
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_core.documents import Document

print("All imports OK")

## 9. Configuration

All tuneable parameters in one place. The exercises at the end ask you to change some of these.

In [ ]:
# ── LLM ────────────────────────────────────────────────
LLM_MODEL = "qwen3.5:9b"
LLM_TEMPERATURE = 0         # Deterministic for reproducibility

# ── Embedding ─────────────────────────────────────────
EMBEDDING_MODEL = "sentence-transformers/all-MiniLM-L6-v2"  # Fast, 384-dim

# ── Chunking ──────────────────────────────────────────
CHUNK_SIZE = 500            # Characters per chunk
CHUNK_OVERLAP = 100         # Overlap between consecutive chunks

# ── Retrieval ──────────────────────────────────────────
TOP_K = 4                   # Number of chunks to retrieve per question

# ── Paths ──────────────────────────────────────────────
PDF_DIR = Path("sample_pdfs")
CHROMA_DIR = Path("chroma_db")

print("Configuration loaded")
print(f"  LLM: {LLM_MODEL}")
print(f"  Embedding: {EMBEDDING_MODEL}")
print(f"  Chunk size: {CHUNK_SIZE} chars, overlap: {CHUNK_OVERLAP} chars")
print(f"  Retrieval top-k: {TOP_K}")

## 10. Create Sample PDF Documents

We generate three PDF files with substantial ML content. This ensures the notebook is **self-contained** — you don't need to download anything.

In your own projects, skip this cell and point `PDF_DIR` at your real PDFs.

In [ ]:
from fpdf import FPDF

PDF_DIR.mkdir(exist_ok=True)


def create_pdf(filename: str, title: str, content: str):
    """Create a simple PDF from text content."""
    pdf = FPDF()
    pdf.set_auto_page_break(auto=True, margin=20)

    # Title page
    pdf.add_page()
    pdf.set_font("Helvetica", "B", 18)
    pdf.cell(0, 15, title, new_x="LMARGIN", new_y="NEXT", align="C")
    pdf.ln(5)

    # Body text
    pdf.set_font("Helvetica", "", 11)
    for paragraph in content.split("\n\n"):
        paragraph = paragraph.strip()
        if not paragraph:
            continue
        if paragraph.startswith("## "):
            pdf.ln(3)
            pdf.set_font("Helvetica", "B", 14)
            pdf.cell(0, 10, paragraph[3:], new_x="LMARGIN", new_y="NEXT")
            pdf.set_font("Helvetica", "", 11)
        else:
            pdf.multi_cell(0, 6, paragraph)
            pdf.ln(2)

    path = PDF_DIR / filename
    pdf.output(str(path))
    return path


# ── PDF 1: ML Fundamentals ────────────────────────────
ML_FUNDAMENTALS = """
## Supervised Learning

Supervised learning is the most common type of machine learning. The algorithm learns from labeled training data, where each example has an input (features) and a known output (label or target). The goal is to learn a mapping function from inputs to outputs that generalizes well to unseen data.

Common supervised learning algorithms include: Linear Regression for continuous targets, Logistic Regression for binary classification, Decision Trees that split data based on feature thresholds, Random Forests that combine multiple decision trees for better generalization, Support Vector Machines (SVM) that find optimal decision boundaries, and Gradient Boosting methods like XGBoost and LightGBM that sequentially correct errors.

The key challenge in supervised learning is obtaining high-quality labeled data. Labels are often expensive to collect (requiring human annotators) and may contain noise or inconsistencies.

## Unsupervised Learning

Unsupervised learning works with unlabeled data. The algorithm discovers hidden patterns or structures without any guidance. This is useful when labels are unavailable or when you want to explore the data structure.

K-Means clustering partitions data into k groups by minimizing within-cluster distances. Each cluster has a centroid, and data points are assigned to the nearest centroid. The algorithm iterates between assigning points and updating centroids until convergence.

DBSCAN (Density-Based Spatial Clustering of Applications with Noise) finds clusters of arbitrary shape by identifying dense regions separated by sparse areas. Unlike K-Means, DBSCAN does not require specifying the number of clusters in advance and can identify outlier points that don't belong to any cluster.

Principal Component Analysis (PCA) reduces dimensionality by projecting data onto the directions of maximum variance. It finds orthogonal axes (principal components) that capture the most information with the fewest dimensions.

## Bias-Variance Tradeoff

The bias-variance tradeoff is a fundamental concept in machine learning. Bias measures how far the model's average predictions are from the true values. High bias means the model is too simple and underfits the data (e.g., fitting a linear model to nonlinear data).

Variance measures how much the model's predictions change when trained on different subsets of the data. High variance means the model is too complex and overfits the training data (e.g., a very deep decision tree that memorizes noise).

The total error decomposes as: Total Error = Bias squared + Variance + Irreducible Noise. The goal is to find the sweet spot where both bias and variance are reasonably low. Regularization techniques (L1/Lasso, L2/Ridge) help control model complexity and reduce variance at the cost of slightly increased bias.

## Cross-Validation

Cross-validation is a technique for estimating model performance on unseen data. K-fold cross-validation splits the data into k equal folds. The model is trained on k-1 folds and evaluated on the remaining fold. This process is repeated k times, each time using a different fold as the validation set.

The final performance estimate is the average across all k folds. This gives a more robust estimate than a single train/test split. Common choices are k=5 or k=10.

Stratified k-fold ensures that each fold has approximately the same class distribution as the full dataset, which is important for imbalanced classification problems. Leave-one-out cross-validation (LOOCV) is a special case where k equals the number of samples, giving the least biased estimate but at high computational cost.

## Feature Engineering

Feature engineering is the process of creating new features or transforming existing ones to improve model performance. Good features can make simple models outperform complex ones on poorly engineered features.

Common techniques include: one-hot encoding for categorical variables, standard scaling or min-max normalization for numerical features, polynomial features to capture nonlinear relationships, interaction features that combine two or more variables, and domain-specific transformations (e.g., log transform for skewed distributions).

Feature selection techniques like mutual information, chi-squared tests, or recursive feature elimination (RFE) identify the most informative features and remove noise, reducing overfitting and training time.
"""

# ── PDF 2: Neural Networks ────────────────────────────
NEURAL_NETWORKS = """
## The Perceptron

The perceptron is the simplest neural network unit. It takes multiple inputs, multiplies each by a weight, sums them up, adds a bias term, and passes the result through an activation function. Mathematically: output = activation(w1*x1 + w2*x2 + ... + wn*xn + bias).

A single perceptron can only learn linearly separable patterns. The XOR problem famously demonstrated this limitation, motivating the development of multi-layer networks.

## Backpropagation

Backpropagation is the core algorithm for training neural networks. It computes gradients of the loss function with respect to each weight using the chain rule of calculus. These gradients indicate how much each weight contributed to the error, allowing the optimizer to update weights in the direction that reduces the loss.

The process works in two passes: the forward pass computes the output by propagating inputs through the network, and the backward pass propagates error gradients from the output layer back to the input layer. Each layer's gradients depend on the gradients of the layer above it, hence the name "backpropagation."

Gradient descent optimizers like SGD (Stochastic Gradient Descent) update weights using: w_new = w_old - learning_rate * gradient. Modern optimizers like Adam combine momentum (using past gradients) with adaptive learning rates for faster convergence.

## Activation Functions

Activation functions introduce nonlinearity into neural networks, enabling them to learn complex patterns. Without nonlinear activations, a multi-layer network would collapse to a single linear transformation.

ReLU (Rectified Linear Unit) outputs max(0, x). It is the most widely used activation because it is computationally cheap and avoids the vanishing gradient problem for positive values. However, ReLU can cause "dying neurons" where units permanently output zero.

Sigmoid outputs values between 0 and 1, making it suitable for binary classification output layers. However, it suffers from vanishing gradients for very large or small inputs, slowing down training in deep networks.

GELU (Gaussian Error Linear Unit) is used in modern transformers like BERT and GPT. It smoothly gates the input based on its value, providing a probabilistic interpretation: the input is multiplied by the probability of being greater than other inputs.

## Convolutional Neural Networks (CNNs)

CNNs are designed for grid-structured data like images. Convolutional layers apply learned filters (kernels) that slide across the input, detecting local patterns like edges, textures, and shapes.

Key CNN concepts: Filters are small weight matrices (e.g., 3x3) that detect specific patterns. Pooling layers reduce spatial dimensions by taking the maximum or average of local regions. Stride controls how far the filter moves between applications. Deeper layers learn increasingly abstract features, progressing from edges to object parts to whole objects.

Important CNN architectures include AlexNet (2012, proved deep CNNs work), VGG (simple repeated blocks), ResNet (skip connections enabling very deep networks, 150+ layers), and EfficientNet (optimized scaling of depth, width, and resolution).

## Recurrent Neural Networks (RNNs)

RNNs process sequential data by maintaining a hidden state that is updated at each time step. This allows them to model temporal dependencies in text, speech, and time series data.

Vanilla RNNs suffer from vanishing and exploding gradients when processing long sequences. LSTM (Long Short-Term Memory) networks solve this with a gating mechanism that controls information flow: the forget gate decides what to discard, the input gate decides what new information to store, and the output gate decides what to output.

GRU (Gated Recurrent Unit) is a simplified variant of LSTM with fewer parameters. It combines the forget and input gates into a single update gate, making it faster to train while achieving comparable performance.

## Transformers

Transformers replaced RNNs for most sequence tasks. The key innovation is the self-attention mechanism, which computes relationships between all positions in a sequence simultaneously (in parallel), rather than sequentially like RNNs.

Self-attention works with three projections: Query (Q), Key (K), and Value (V). For each position, attention scores are computed as the dot product of Q with all K vectors, scaled and softmaxed, then used to weight the V vectors. This allows each token to attend to any other token regardless of distance.

Multi-head attention runs several attention computations in parallel with different learned projections, allowing the model to attend to different types of relationships simultaneously (e.g., syntactic, semantic).

The original Transformer architecture (Vaswani et al., 2017) uses an encoder-decoder structure. BERT (Devlin et al., 2019) uses only the encoder for classification and understanding tasks. GPT (Radford et al., 2018) uses only the decoder for text generation. Modern LLMs like GPT-4, LLaMA, and Qwen are decoder-only transformer models trained on massive text corpora.
"""

# ── PDF 3: ML Deployment Guide ────────────────────────
DEPLOYMENT_GUIDE = """
## Model Serving

Model serving is the process of making a trained ML model available for real-time or batch predictions. Common serving frameworks include TensorFlow Serving, TorchServe, Triton Inference Server, and simple REST APIs built with FastAPI or Flask.

For real-time serving, latency is critical. Techniques to reduce inference latency include model quantization (converting float32 weights to int8, reducing model size by 4x with minimal accuracy loss), model pruning (removing low-magnitude weights), and ONNX export (converting models to an optimized format that runs on any hardware).

Batch serving processes large datasets offline, typically on a schedule. This is suitable for recommendation systems, report generation, and any use case where real-time predictions are not required.

## Model Monitoring

Model monitoring detects when a deployed model's performance degrades over time. The two main types of drift are:

Data drift occurs when the input data distribution changes. For example, if a model was trained on summer data and deployed in winter, seasonal patterns will cause different input distributions.

Concept drift occurs when the relationship between inputs and outputs changes. For example, user preferences may shift over time, making yesterday's recommendations less relevant today.

Monitoring techniques include tracking prediction distributions over time, comparing feature distributions against training data baselines, logging confidence scores and flagging low-confidence predictions, and measuring actual vs predicted outcomes when ground truth labels become available.

## A/B Testing for ML Models

A/B testing compares a new model (variant B) against the current production model (variant A) by randomly routing a percentage of traffic to each. Key principles: use a statistically meaningful sample size, run the test long enough to capture temporal patterns, measure business metrics (not just model metrics), and account for network effects in social platforms.

Shadow deployment runs the new model alongside the existing one without serving its predictions to users. This allows measuring performance differences without risk. Canary deployment gradually increases traffic to the new model (1%, 5%, 25%, 50%, 100%) while monitoring for regressions.

## MLOps Best Practices

MLOps applies DevOps principles to machine learning. Key practices include:

Version control everything: code, data, models, and configuration. Tools like DVC (Data Version Control) track large data files and model artifacts alongside Git.

Automate the ML pipeline: data ingestion, feature engineering, training, evaluation, and deployment should be reproducible with a single command. Tools like Kubeflow, Airflow, and MLflow Pipelines orchestrate these steps.

Track experiments with MLflow or Weights & Biases to log hyperparameters, metrics, and artifacts for every training run. This enables reproducibility and comparison.

Containerize models with Docker to ensure consistent environments across development, testing, and production. Kubernetes can auto-scale model serving pods based on request volume.

Implement CI/CD for ML: automated testing of data quality, model performance thresholds, and integration tests before deploying new model versions.
"""

# ── Create the PDFs ───────────────────────────────────
pdfs = [
    ("ml_fundamentals.pdf", "Machine Learning Fundamentals", ML_FUNDAMENTALS),
    ("neural_networks.pdf", "Neural Networks Deep Dive", NEURAL_NETWORKS),
    ("deployment_guide.pdf", "ML Deployment Guide", DEPLOYMENT_GUIDE),
]

for fname, title, content in pdfs:
    path = create_pdf(fname, title, content)
    size_kb = path.stat().st_size / 1024
    print(f"  Created: {path} ({size_kb:.1f} KB)")

print(f"\n{len(pdfs)} sample PDFs created in '{PDF_DIR}/'")

## 11. Load PDF Documents

We use LangChain's `PyPDFLoader` to read each PDF file. Each page becomes a `Document` object with:
- `.page_content` — the extracted text
- `.metadata` — source file name, page number, etc.

The metadata travels with the text through the entire pipeline, so we can **cite sources** in our answers.

In [ ]:
# ── Load all PDFs from the directory ──────────────────
all_docs = []

pdf_files = sorted(PDF_DIR.glob("*.pdf"))
print(f"Found {len(pdf_files)} PDF files\n")

for pdf_path in pdf_files:
    loader = PyPDFLoader(str(pdf_path))
    pages = loader.load()
    all_docs.extend(pages)
    print(f"  {pdf_path.name}: {len(pages)} pages loaded")

print(f"\nTotal documents (pages): {len(all_docs)}")

## 12. Inspect Loaded Documents

Before processing, always inspect what your document loader actually extracted. Common issues include garbled text, missing content, and encoding problems.

In [ ]:
# ── Inspect the first document ─────────────────────────
doc = all_docs[0]
print("Document object attributes:")
print(f"  Type: {type(doc).__name__}")
print(f"  Metadata: {doc.metadata}")
print(f"  Content length: {len(doc.page_content)} characters")
print(f"  Content preview:\n")
print(doc.page_content[:500])
print("...")

# ── Summary of all documents ───────────────────────────
print("\n" + "=" * 50)
print("All loaded documents:")
for i, d in enumerate(all_docs):
    source = Path(d.metadata.get("source", "unknown")).name
    page = d.metadata.get("page", "?")
    chars = len(d.page_content)
    words = len(d.page_content.split())
    print(f"  [{i}] {source} p.{page} — {chars:,} chars, {words:,} words")

## 13. Split Documents into Chunks

### Why chunk?

LLMs have limited context windows, and embedding models work best on smaller text passages. We split documents into overlapping chunks so that:
1. Each chunk fits within the embedding model's token limit
2. Retrieval returns focused, relevant passages (not entire pages)
3. Overlap ensures we don't lose information at chunk boundaries

### `RecursiveCharacterTextSplitter`

This splitter tries to keep semantically related text together. It splits by paragraph first (`\n\n`), then by sentence (`\n`), then by word (` `), then by character — stopping as soon as chunks are small enough. This preserves meaning better than splitting at fixed character positions.

In [ ]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
    length_function=len,
    separators=["\n\n", "\n", ". ", " ", ""],
)

chunks = splitter.split_documents(all_docs)

print(f"Split {len(all_docs)} pages into {len(chunks)} chunks")
print(f"Chunk size: {CHUNK_SIZE} chars, overlap: {CHUNK_OVERLAP} chars")

# ── Show chunk size distribution ──────────────────────
sizes = [len(c.page_content) for c in chunks]
print(f"\nChunk size stats:")
print(f"  Min: {min(sizes)} chars")
print(f"  Max: {max(sizes)} chars")
print(f"  Mean: {sum(sizes)/len(sizes):.0f} chars")

# ── Show a few example chunks ─────────────────────────
print("\n" + "=" * 50)
for i in [0, len(chunks)//2, len(chunks)-1]:
    c = chunks[i]
    source = Path(c.metadata.get("source", "?")).name
    print(f"\nChunk [{i}] from {source} p.{c.metadata.get('page', '?')} ({len(c.page_content)} chars):")
    print(c.page_content[:200] + "...")

## 14. Create Embeddings & Vector Store

### What are embeddings?

An embedding model converts text into a **dense numerical vector** (e.g., 384 dimensions for `all-MiniLM-L6-v2`). Texts with similar meaning produce vectors that are close together in this high-dimensional space.

### Why ChromaDB?

ChromaDB is a lightweight vector database that:
- Stores embeddings and their associated text/metadata
- Performs fast similarity search (cosine distance)
- Works locally, no server setup needed
- Persists to disk for reuse across sessions

In [ ]:
# ── Initialize embedding model (runs locally on CPU/GPU) ─
print(f"Loading embedding model: {EMBEDDING_MODEL}")
t0 = time.perf_counter()

embeddings = HuggingFaceEmbeddings(
    model_name=EMBEDDING_MODEL,
    model_kwargs={"device": "cpu"},   # Use "cuda" if you have GPU memory to spare
)

load_time = time.perf_counter() - t0
print(f"Embedding model loaded in {load_time:.1f}s")

# ── Quick test: embed a single sentence ───────────────
test_vec = embeddings.embed_query("What is machine learning?")
print(f"Embedding dimension: {len(test_vec)}")
print(f"Sample values: [{test_vec[0]:.4f}, {test_vec[1]:.4f}, {test_vec[2]:.4f}, ...]")

In [ ]:
# ── Build the vector store ────────────────────────────
# Clean up any previous ChromaDB to avoid stale data
import shutil
if CHROMA_DIR.exists():
    shutil.rmtree(CHROMA_DIR)

print(f"Embedding {len(chunks)} chunks into ChromaDB...")
t0 = time.perf_counter()

vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    persist_directory=str(CHROMA_DIR),
)

embed_time = time.perf_counter() - t0
print(f"Vector store built in {embed_time:.1f}s")
print(f"Collection size: {vectorstore._collection.count()} vectors")
print(f"Persisted to: {CHROMA_DIR}/")

### 14b. Test Retrieval

Before building the full RAG chain, verify that the vector store returns sensible results for a test query.

In [ ]:
# ── Test retrieval ────────────────────────────────────
test_query = "What is the bias-variance tradeoff?"
results = vectorstore.similarity_search_with_score(test_query, k=3)

print(f"Query: '{test_query}'")
print(f"Top {len(results)} results:\n")

for i, (doc, score) in enumerate(results):
    source = Path(doc.metadata.get("source", "?")).name
    page = doc.metadata.get("page", "?")
    print(f"  [{i+1}] Score: {score:.4f} | {source} p.{page}")
    print(f"      {doc.page_content[:150]}...")
    print()

## 15. Baseline — Naive Prompting (No RAG)

Before building the RAG pipeline, let's see how the LLM performs **without** document context. This establishes a baseline and demonstrates exactly why RAG is needed.

We ask questions about **specific details from our PDFs**. The LLM might give a reasonable-sounding general answer, but it cannot cite our documents and may hallucinate specific details.

In [ ]:
# ── Initialize the LLM ────────────────────────────────
llm = ChatOllama(model=LLM_MODEL, temperature=LLM_TEMPERATURE)
print(f"LLM: {LLM_MODEL} (temperature={LLM_TEMPERATURE})")

In [ ]:
# ── Naive prompting: just ask the LLM directly ───────
NAIVE_QUESTIONS = [
    "What does our deployment guide say about A/B testing for ML models?",
    "According to the documents, what are the main differences between LSTM and GRU?",
    "What feature engineering techniques are recommended in our ML fundamentals guide?",
]

def clean_response(text: str) -> str:
    """Remove thinking tags from model output."""
    if "<think>" in text:
        text = text.split("</think>")[-1].strip()
    return text

print("NAIVE PROMPTING (no document context)")
print("=" * 60)

naive_answers = {}
for q in NAIVE_QUESTIONS:
    print(f"\nQ: {q}")
    resp = llm.invoke([
        SystemMessage(content="Answer the question concisely. If you don't know, say so."),
        HumanMessage(content=q)
    ])
    answer = clean_response(resp.content)
    naive_answers[q] = answer
    print(f"A: {answer[:300]}")
    print("-" * 40)

### Observation

Notice that the LLM:
- **Cannot reference "our documents"** — it has no access to them
- **May give generic correct answers** from its training data (it knows ML broadly)
- **Cannot cite page numbers or specific sources**
- **May hallucinate details** that aren't in our guides

The key question is: *did the answer come from our actual documents, or from the LLM's general knowledge?* Without RAG, we can't tell.

## 16. Main Workflow — RAG Pipeline with Citations

Now we build the actual RAG pipeline. For each question:
1. **Retrieve** the top-k most relevant chunks from ChromaDB
2. **Format** the chunks into a context string with source citations
3. **Prompt** the LLM with context + question
4. **Return** a grounded answer with source references

In [ ]:
# ── Create the retriever ──────────────────────────────
retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": TOP_K}
)

# ── RAG prompt template ───────────────────────────────
RAG_SYSTEM_PROMPT = """You are a helpful assistant that answers questions based ONLY on the provided context.

Rules:
1. Answer ONLY using information from the context below
2. If the context doesn't contain the answer, say "I cannot find this information in the provided documents."
3. Cite your sources using [Source: filename, Page X] format
4. Be concise but thorough
5. Do not make up information not present in the context"""

RAG_USER_TEMPLATE = """Context from documents:
---
{context}
---

Question: {question}

Answer (with source citations):"""

print("Retriever and prompt template configured")
print(f"Search type: similarity, top-k: {TOP_K}")

In [ ]:
def ask_rag(question: str, verbose: bool = True) -> dict:
    """
    Full RAG pipeline: retrieve → format context → generate answer.
    Returns dict with question, answer, sources, and retrieved chunks.
    """
    # Step 1: Retrieve relevant chunks
    retrieved_docs = retriever.invoke(question)

    # Step 2: Format context with source attribution
    context_parts = []
    sources = []
    for i, doc in enumerate(retrieved_docs):
        source_name = Path(doc.metadata.get("source", "unknown")).name
        page_num = doc.metadata.get("page", "?")
        source_ref = f"{source_name}, Page {page_num}"
        sources.append(source_ref)
        context_parts.append(f"[Source: {source_ref}]\n{doc.page_content}")

    context = "\n\n".join(context_parts)

    # Step 3: Generate answer using LLM
    messages = [
        SystemMessage(content=RAG_SYSTEM_PROMPT),
        HumanMessage(content=RAG_USER_TEMPLATE.format(
            context=context, question=question
        ))
    ]
    response = llm.invoke(messages)
    answer = clean_response(response.content)

    result = {
        "question": question,
        "answer": answer,
        "sources": list(set(sources)),
        "num_chunks": len(retrieved_docs),
        "chunks": retrieved_docs,
    }

    if verbose:
        print(f"Q: {question}")
        print(f"\nA: {answer}")
        print(f"\nSources: {', '.join(result['sources'])}")
        print(f"Chunks retrieved: {result['num_chunks']}")

    return result

print("ask_rag() function defined")

## 17. Run the RAG Pipeline

Let's ask the **same questions** we used for naive prompting, plus a few more. Compare the answers to see how RAG improves grounding and accuracy.

In [ ]:
QUESTIONS = [
    # Same as naive baseline
    "What does our deployment guide say about A/B testing for ML models?",
    "According to the documents, what are the main differences between LSTM and GRU?",
    "What feature engineering techniques are recommended in our ML fundamentals guide?",
    # Additional questions
    "How does the self-attention mechanism work in transformers?",
    "What are the key MLOps best practices mentioned in the deployment guide?",
    "Explain the difference between data drift and concept drift.",
]

rag_results = []
for q in QUESTIONS:
    print("\n" + "=" * 60)
    result = ask_rag(q)
    rag_results.append(result)

## 18. Evaluation — Naive Prompting vs RAG

Now we directly compare the two approaches on the same questions. This is the core insight of this notebook: **RAG produces grounded, citable answers** while naive prompting produces generic ones.

In [ ]:
print("SIDE-BY-SIDE COMPARISON: Naive vs RAG")
print("=" * 70)

for q in NAIVE_QUESTIONS:
    rag_result = next((r for r in rag_results if r["question"] == q), None)
    if not rag_result:
        continue

    print(f"\nQ: {q}")
    print(f"\n  NAIVE (no context):")
    print(f"  {naive_answers[q][:250]}")
    print(f"\n  RAG (with document context):")
    print(f"  {rag_result['answer'][:250]}")
    print(f"  Sources: {', '.join(rag_result['sources'])}")
    print("-" * 70)

In [ ]:
# ── LLM-as-judge: which answer is better? ─────────────
print("LLM-AS-JUDGE EVALUATION")
print("=" * 60)

eval_results = []

for q in NAIVE_QUESTIONS:
    rag_result = next((r for r in rag_results if r["question"] == q), None)
    if not rag_result:
        continue

    judge_prompt = f"""Compare these two answers to the question: "{q}"

Answer A (no document context):
{naive_answers[q][:400]}

Answer B (with retrieved document context):
{rag_result['answer'][:400]}

Which answer is better for a user who wants information FROM SPECIFIC DOCUMENTS?
Rate each on a 1-10 scale for: groundedness, specificity, and source citation.
Return your judgment in 2-3 sentences. No thinking tags."""

    resp = llm.invoke([HumanMessage(content=judge_prompt)])
    judgment = clean_response(resp.content)

    print(f"\nQ: {q[:60]}...")
    print(f"Judge: {judgment[:300]}")
    print("-" * 40)

### Key Differences

| Aspect | Naive Prompting | RAG |
|--------|----------------|-----|
| **Source of knowledge** | LLM's training data | Your actual documents |
| **Groundedness** | May hallucinate details | Answers grounded in retrieved text |
| **Citations** | Cannot cite sources | Cites filename and page number |
| **Private data** | No access | Full access to your PDFs |
| **Updatable** | Requires fine-tuning | Just add new PDFs |
| **Cost per query** | 1 LLM call | 1 embedding + 1 retrieval + 1 LLM call |

## 19. Inspect Retrieved Chunks in Detail

For debugging and understanding how RAG works, it's important to see exactly which chunks were retrieved and why. This helps diagnose issues like irrelevant retrieval or missing context.

In [ ]:
# ── Detailed retrieval inspection ─────────────────────
inspect_q = "How does the self-attention mechanism work in transformers?"
print(f"Query: '{inspect_q}'\n")

results_with_scores = vectorstore.similarity_search_with_score(inspect_q, k=6)

for i, (doc, score) in enumerate(results_with_scores):
    source = Path(doc.metadata.get("source", "?")).name
    page = doc.metadata.get("page", "?")
    relevant = "✅" if i < TOP_K else "❌ (below top-k cutoff)"
    print(f"Rank {i+1} | Score: {score:.4f} | {source} p.{page} {relevant}")
    print(f"  {doc.page_content[:200]}")
    print()

print(f"Note: Lower scores = more similar (L2 distance in Chroma).")
print(f"Only the top {TOP_K} chunks are sent to the LLM.")

## 20. Multi-Question Session with Conversation History

In real applications, users ask follow-up questions. Here we build a simple multi-turn Q&A session that maintains context.

In [ ]:
def ask_session(questions: list[str]) -> list[dict]:
    """Run a multi-question session, printing results as we go."""
    results = []
    for i, q in enumerate(questions, 1):
        print(f"\n{'─' * 50}")
        print(f"Question {i}/{len(questions)}")
        r = ask_rag(q)
        results.append(r)
    return results

session_questions = [
    "What is the bias-variance tradeoff?",
    "What regularization techniques help with it?",
    "How does cross-validation estimate model performance?",
    "What is the difference between shadow and canary deployment?",
]

print("MULTI-QUESTION SESSION")
print("=" * 50)
session_results = ask_session(session_questions)

## 21. Error Analysis & Limitations

RAG isn't perfect. Let's test edge cases to understand where it fails and why.

In [ ]:
# ── Edge case 1: Question about content NOT in the documents ──
print("EDGE CASE 1: Question about content not in our PDFs")
print("=" * 50)
r1 = ask_rag("What is the company's vacation policy?")

print("\n" + "=" * 50)
print("EDGE CASE 2: Ambiguous question that spans multiple documents")
print("=" * 50)
r2 = ask_rag("Tell me everything you know about neural networks and how to deploy them")

print("\n" + "=" * 50)
print("EDGE CASE 3: Very specific question requiring exact detail")
print("=" * 50)
r3 = ask_rag("What activation function is used in BERT and GPT?")

### Limitations Analysis

| Limitation | Explanation | Mitigation |
|------------|-------------|------------|
| **Chunking artifacts** | Important context may be split across chunk boundaries | Increase overlap or use semantic chunking |
| **Retrieval misses** | Semantically relevant chunks may not be in top-k | Increase k, use hybrid search (BM25 + vector) |
| **Context window limits** | Too many chunks overflow the LLM's context | Summarize or re-rank retrieved chunks |
| **No cross-document reasoning** | Each chunk is retrieved independently | Use graph RAG or multi-hop retrieval |
| **Embedding model quality** | Small models may miss nuanced semantics | Use larger models (e.g., BGE-large) |
| **OCR quality for scanned PDFs** | PyPDF can't read images or scanned text | Use OCR tools like Tesseract or PaddleOCR |
| **Hallucination despite context** | LLM may still hallucinate details | Use stricter prompts, lower temperature |

## 22. Common Mistakes

| Mistake | Why It's Wrong | What to Do Instead |
|---------|---------------|-------------------|
| **Chunks too large (2000+ chars)** | Retrieval returns vague, unfocused passages; wastes context window | Use 300–800 char chunks with 50–150 overlap |
| **Chunks too small (<100 chars)** | Fragments lack enough context to be useful | Keep chunks large enough for a complete thought |
| **No overlap between chunks** | Information at boundaries is lost | Use 10–30% overlap (e.g., 100 overlap for 500 chunk) |
| **Not cleaning up ChromaDB between runs** | Stale vectors from previous experiments pollute results | Delete and recreate the collection when reindexing |
| **Using the wrong embedding model for retrieval vs indexing** | Vectors from different models aren't comparable | Always use the same model for both |
| **Stuffing all retrieved chunks without limit** | Overflows LLM context window, gets truncated unpredictably | Limit to top-k and monitor total token count |
| **Ignoring metadata** | Lose the ability to cite sources or filter by document | Always propagate metadata through the pipeline |
| **Not testing retrieval independently** | Can't tell if bad answers come from retrieval or generation | Test `similarity_search` separately before building the chain |

## 23. How to Improve This Project

| Improvement | Difficulty | Expected Impact |
|-------------|------------|----------------|
| **Hybrid search** (BM25 keyword + vector similarity) | Medium | Better recall for exact-match queries |
| **Re-ranking** with a cross-encoder model | Medium | More precise top-k selection |
| **Semantic chunking** (split by topic, not character count) | Medium | More coherent chunks |
| **Multi-vector retrieval** (summary + full chunk) | Medium | Better retrieval for broad questions |
| **Conversational memory** (track chat history) | Easy | Better follow-up question handling |
| **Larger embedding model** (e.g., `BAAI/bge-large-en-v1.5`) | Easy | Better semantic matching |
| **Query expansion** (rephrase question before retrieval) | Medium | Catches more relevant chunks |
| **Graph RAG** (extract entities + relationships) | Hard | Cross-document reasoning |
| **Evaluation framework** (RAGAS, LangSmith) | Medium | Systematic quality measurement |

## 24. Production Improvement Ideas

To turn this learning notebook into a production system:

1. **Persistent vector store** — Use PostgreSQL + pgvector or Pinecone instead of local ChromaDB
2. **Document ingestion pipeline** — Watch a folder for new PDFs, auto-index on upload
3. **User authentication** — Track who asks what, for audit and personalization
4. **Feedback loop** — Let users rate answers; use ratings to improve retrieval
5. **Streaming responses** — Stream LLM output token-by-token for better UX
6. **Caching** — Cache embeddings and frequent query results to reduce latency
7. **Rate limiting** — Protect the LLM backend from abuse
8. **Metrics dashboard** — Track retrieval relevance, answer quality, and latency over time
9. **Multi-format support** — Add loaders for DOCX, HTML, Markdown, and web pages
10. **Deploy as API** — Wrap in FastAPI with `/upload`, `/ask`, and `/sources` endpoints

## 25. Exercises — Try These Yourself

### Exercise 1: Change Chunk Size

Go back to the Configuration cell and change `CHUNK_SIZE` to `250` and then to `1000`. Re-run the notebook and compare:
- How many chunks are created?
- Does retrieval quality change?
- Are answers more or less detailed?

**Hypothesis:** Smaller chunks → more focused retrieval but less context per chunk. Larger chunks → more context but may include irrelevant text.

In [ ]:
# ── Exercise 1: Chunk size comparison ─────────────────
# Try different chunk sizes and see how many chunks you get
test_question = "What is the bias-variance tradeoff?"

for size in [250, 500, 1000]:
    test_splitter = RecursiveCharacterTextSplitter(
        chunk_size=size, chunk_overlap=size // 5
    )
    test_chunks = test_splitter.split_documents(all_docs)
    avg_len = sum(len(c.page_content) for c in test_chunks) / len(test_chunks)
    print(f"  Chunk size {size:>5}: {len(test_chunks):>3} chunks, avg {avg_len:.0f} chars")

### Exercise 2: Change the Embedding Model

Try replacing `all-MiniLM-L6-v2` (384-dim, fast) with a larger model. How does it affect retrieval quality?

In [ ]:
# ── Exercise 2: Compare embedding models ──────────────
# Uncomment one alternative model to try:
ALTERNATIVE_MODELS = [
    # "sentence-transformers/all-mpnet-base-v2",     # 768-dim, higher quality
    # "BAAI/bge-small-en-v1.5",                      # 384-dim, newer architecture
    # "BAAI/bge-large-en-v1.5",                      # 1024-dim, best quality (slower)
]

# To compare, uncomment below and pick a model:
# alt_model = "BAAI/bge-small-en-v1.5"
# alt_embeddings = HuggingFaceEmbeddings(model_name=alt_model)
# alt_vectorstore = Chroma.from_documents(chunks, alt_embeddings)
# alt_results = alt_vectorstore.similarity_search_with_score(test_question, k=3)
# for i, (doc, score) in enumerate(alt_results):
#     print(f"  [{i+1}] Score: {score:.4f} | {doc.page_content[:100]}...")

print("Exercise 2: Uncomment the code above to compare embedding models")
print("Larger models are slower but capture more semantic nuance")

### Exercise 3: Change the Retriever

Try changing `search_type` and `k` to see how retrieval behavior changes.

In [ ]:
# ── Exercise 3: Compare retriever settings ────────────
test_q = "How does the self-attention mechanism work?"

for k in [2, 4, 8]:
    results = vectorstore.similarity_search(test_q, k=k)
    sources = set()
    for r in results:
        sources.add(Path(r.metadata["source"]).name)
    total_context_chars = sum(len(r.page_content) for r in results)
    print(f"  k={k}: {len(results)} chunks, {total_context_chars:,} chars, "
          f"from {len(sources)} docs: {sources}")

print("\nTrade-off: higher k = more context but may include noise")
print("Lower k = focused but may miss important information")

### Mini Challenge

1. **Add a new PDF:** Write content about a topic you know well (cooking recipes, sports rules, etc.). Add it to `sample_pdfs/`, rebuild the vector store, and ask questions about your new content.

2. **Build a guardrail:** Modify `ask_rag()` to check if the retrieved chunks have a minimum similarity score. If all chunks score poorly (distance > 1.5), refuse to answer instead of hallucinating.

3. **Implement MMR retrieval:** Change `search_type` to `"mmr"` (Maximal Marginal Relevance). MMR balances relevance with diversity — it avoids retrieving near-duplicate chunks. Compare the results.

4. **Source filtering:** Modify `ask_rag()` to accept an optional `source_filter` parameter that restricts retrieval to chunks from a specific PDF only.

## 26. Cleanup

In [ ]:
# ── Summary statistics ────────────────────────────────
print("PDF Q&A ASSISTANT — SESSION SUMMARY")
print("=" * 50)
print(f"PDFs loaded: {len(pdf_files)}")
print(f"Pages extracted: {len(all_docs)}")
print(f"Chunks created: {len(chunks)}")
print(f"Chunk size: {CHUNK_SIZE} chars, overlap: {CHUNK_OVERLAP}")
print(f"Embedding model: {EMBEDDING_MODEL}")
print(f"Vector store size: {vectorstore._collection.count()} vectors")
print(f"LLM: {LLM_MODEL}")
print(f"Questions answered (RAG): {len(rag_results)}")
print(f"Questions answered (naive baseline): {len(naive_answers)}")

# Uncomment to delete the ChromaDB and sample PDFs:
# import shutil
# shutil.rmtree(CHROMA_DIR, ignore_errors=True)
# shutil.rmtree(PDF_DIR, ignore_errors=True)
# print("\nCleaned up ChromaDB and sample PDFs")

## 27. Key Takeaways

| # | Takeaway |
|---|----------|
| 1 | **RAG = Retrieval + Generation.** The retriever finds relevant context; the LLM generates a grounded answer |
| 2 | **Chunking matters.** Too large → unfocused retrieval. Too small → fragmented context. 300–800 chars with 10–30% overlap is a good starting point |
| 3 | **Embeddings encode meaning.** Similar questions and answers end up as nearby vectors, enabling semantic search |
| 4 | **Always test retrieval separately.** Bad RAG answers often come from bad retrieval, not bad generation |
| 5 | **Metadata is essential.** Propagating source + page number enables citations and filtering |
| 6 | **Naive prompting cannot access private documents.** RAG is the standard solution for document Q&A |
| 7 | **The prompt template is critical.** Instructing the LLM to "only use provided context" dramatically reduces hallucination |
| 8 | **RAG is modular.** You can swap the embedding model, vector store, LLM, and chunking strategy independently |

---

*This notebook is part of the [Machine Learning Projects](https://github.com/pypi-ahmad/Machine-Learning-Projects) repository.*